### Setup & Installation
First, let's install the required libraries like `sentencepiece` which might not be installed in the environment by default.

In [1]:
!pip install sentencepiece transformers gensim spacy nltk
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('tagsets')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 36.0 MB/s  0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package tagsets to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets.zip.


True

### 5.1. BPE-based Tokenizer
We will implement BPE-based tokenization using:
* **a. Pretrained model:** Using `GPT2Tokenizer` from Hugging Face.
* **b. Without using pretrained model:** Implementing a basic BPE algorithm from scratch on the input text.

In [2]:
# Read input text for 5.1
try:
    with open('/content/input_sub_word_data.txt', 'r', encoding='utf-8') as f:
        input_text = f.read().strip()
except FileNotFoundError:
    # Fallback placeholder in case the file is not yet present
    input_text = "Natural language processing (NLP) is a subfield of computer science, information engineering, and artificial intelligence."
    with open('/content/input_sub_word_data.txt', 'w', encoding='utf-8') as f:
        f.write(input_text)

print("Sample input text preview:", input_text[:100])

Sample input text preview: Natural language processing is a branch of artificial intelligence that
enables computers to underst


In [3]:
# 5.1.a Using Pretrained BPE Model (GPT-2)
from transformers import GPT2Tokenizer

tokenizer_bpe_pre = GPT2Tokenizer.from_pretrained('gpt2')
tokens_pre = tokenizer_bpe_pre.tokenize(input_text)
token_ids_pre = tokenizer_bpe_pre.convert_tokens_to_ids(tokens_pre)

print("--- 5.1.a Pretrained GPT-2 BPE Tokenizer ---")
print(f"Original Length: {len(input_text)} chars | Tokens Count: {len(tokens_pre)}")
print("Tokens & IDs (showing first 30):")
for tok, tid in list(zip(tokens_pre, token_ids_pre))[:30]:
    print(f"  Token: {repr(tok):<15} | ID: {tid}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1348 > 1024). Running this sequence through the model will result in indexing errors


--- 5.1.a Pretrained GPT-2 BPE Tokenizer ---
Original Length: 6682 chars | Tokens Count: 1348
Tokens & IDs (showing first 30):
  Token: 'Natural'       | ID: 35364
  Token: 'Ġlanguage'     | ID: 3303
  Token: 'Ġprocessing'   | ID: 7587
  Token: 'Ġis'           | ID: 318
  Token: 'Ġa'            | ID: 257
  Token: 'Ġbranch'       | ID: 8478
  Token: 'Ġof'           | ID: 286
  Token: 'Ġartificial'   | ID: 11666
  Token: 'Ġintelligence' | ID: 4430
  Token: 'Ġthat'         | ID: 326
  Token: 'Ċ'             | ID: 198
  Token: 'en'            | ID: 268
  Token: 'ables'         | ID: 2977
  Token: 'Ġcomputers'    | ID: 9061
  Token: 'Ġto'           | ID: 284
  Token: 'Ġunderstand'   | ID: 1833
  Token: ','             | ID: 11
  Token: 'Ġinterpret'    | ID: 6179
  Token: ','             | ID: 11
  Token: 'Ġand'          | ID: 290
  Token: 'Ġgenerate'     | ID: 7716
  Token: 'Ġhuman'        | ID: 1692
  Token: 'Ġlanguage'     | ID: 3303
  Token: '.'             | ID: 13
  Token: 'Ċ'         

In [4]:
# 5.1.b Custom BPE Tokenizer from Scratch (Without Pretrained Model)
from collections import defaultdict, Counter
import re

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Initialize vocabulary
words = input_text.split()
vocab = Counter([' '.join(list(w)) + ' </w>' for w in words])

# Run BPE merges
num_merges = 50
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)

# Build token mapping
unique_subwords = set()
for word in vocab:
    for subword in word.split():
        unique_subwords.add(subword)

subword_to_id = {subword: idx for idx, subword in enumerate(sorted(unique_subwords))}

print("--- 5.1.b Custom BPE Tokenizer (From Scratch) ---")
print(f"Unique Subwords Created: {len(subword_to_id)}")
print("Tokens & IDs:")
for subword, idx in list(subword_to_id.items())[:30]:
    print(f"  Token: {repr(subword):<15} | ID: {idx}")

--- 5.1.b Custom BPE Tokenizer (From Scratch) ---
Unique Subwords Created: 92
Tokens & IDs:
  Token: ',</w>'         | ID: 0
  Token: '-'             | ID: 1
  Token: '.</w>'         | ID: 2
  Token: '</w>'          | ID: 3
  Token: 'A'             | ID: 4
  Token: 'B'             | ID: 5
  Token: 'C'             | ID: 6
  Token: 'D'             | ID: 7
  Token: 'E'             | ID: 8
  Token: 'F'             | ID: 9
  Token: 'H'             | ID: 10
  Token: 'I'             | ID: 11
  Token: 'L'             | ID: 12
  Token: 'M'             | ID: 13
  Token: 'N'             | ID: 14
  Token: 'P'             | ID: 15
  Token: 'R'             | ID: 16
  Token: 'S'             | ID: 17
  Token: 'T'             | ID: 18
  Token: 'W'             | ID: 19
  Token: 'a'             | ID: 20
  Token: 'a</w>'         | ID: 21
  Token: 'al'            | ID: 22
  Token: 'al</w>'        | ID: 23
  Token: 'an'            | ID: 24
  Token: 'an</w>'        | ID: 25
  Token: 'and</w>'       | ID: 26


### 5.2. SentencePiece Tokenizer
We will implement subword-level tokenization using SentencePiece:
* **a. Using pretrained model:** Using a pretrained model like `xlnet-base-cased`.
* **b. Without using pretrained model:** Training a fresh SentencePiece model on `/content/input_sub_word_data.txt`.

In [5]:
# 5.2.a Pretrained SentencePiece (XLNet)
from transformers import XLNetTokenizer

tokenizer_sp_pre = XLNetTokenizer.from_pretrained('xlnet-base-cased')
tokens_sp = tokenizer_sp_pre.tokenize(input_text)
token_ids_sp = tokenizer_sp_pre.convert_tokens_to_ids(tokens_sp)

print("--- 5.2.a Pretrained SentencePiece (XLNet) ---")
print("Tokens & IDs (showing first 30):")
for tok, tid in list(zip(tokens_sp, token_ids_sp))[:30]:
    print(f"  Token: {repr(tok):<15} | ID: {tid}")

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

--- 5.2.a Pretrained SentencePiece (XLNet) ---
Tokens & IDs (showing first 30):
  Token: '▁Natural'      | ID: 6421
  Token: '▁language'     | ID: 1243
  Token: '▁processing'   | ID: 4218
  Token: '▁is'           | ID: 27
  Token: '▁a'            | ID: 24
  Token: '▁branch'       | ID: 3709
  Token: '▁of'           | ID: 20
  Token: '▁artificial'   | ID: 8298
  Token: '▁intelligence' | ID: 2503
  Token: '▁that'         | ID: 29
  Token: '▁enables'      | ID: 8350
  Token: '▁computers'    | ID: 3668
  Token: '▁to'           | ID: 22
  Token: '▁understand'   | ID: 1111
  Token: ','             | ID: 19
  Token: '▁interpret'    | ID: 11001
  Token: ','             | ID: 19
  Token: '▁and'          | ID: 21
  Token: '▁generate'     | ID: 5324
  Token: '▁human'        | ID: 594
  Token: '▁language'     | ID: 1243
  Token: '.'             | ID: 9
  Token: '▁Natural'      | ID: 6421
  Token: '▁language'     | ID: 1243
  Token: '▁processing'   | ID: 4218
  Token: '▁combines'     | ID: 12854
  

In [6]:
# 5.2.b Train SentencePiece Model from Scratch
import sentencepiece as spm

# Train the SentencePiece model on the input file
spm.SentencePieceTrainer.train(
    input='/content/input_sub_word_data.txt',
    model_prefix='m_user',
    vocab_size=100,
    model_type='unigram', # unigram is default for SentencePiece
    character_coverage=1.0
)

# Load trained model
sp = spm.SentencePieceProcessor(model_file='m_user.model')
tokens_sp_sc = sp.encode_as_pieces(input_text)
token_ids_sp_sc = sp.encode_as_ids(input_text)

print("--- 5.2.b Custom Trained SentencePiece ---")
print("Tokens & IDs (showing first 30):")
for tok, tid in list(zip(tokens_sp_sc, token_ids_sp_sc))[:30]:
    print(f"  Token: {repr(tok):<15} | ID: {tid}")

--- 5.2.b Custom Trained SentencePiece ---
Tokens & IDs (showing first 30):
  Token: '▁'             | ID: 3
  Token: 'N'             | ID: 83
  Token: 'a'             | ID: 6
  Token: 't'             | ID: 9
  Token: 'u'             | ID: 12
  Token: 'r'             | ID: 7
  Token: 'al'            | ID: 24
  Token: '▁language'     | ID: 38
  Token: '▁'             | ID: 3
  Token: 'p'             | ID: 16
  Token: 'r'             | ID: 7
  Token: 'o'             | ID: 11
  Token: 'c'             | ID: 15
  Token: 'e'             | ID: 4
  Token: 's'             | ID: 5
  Token: 's'             | ID: 5
  Token: 'ing'           | ID: 21
  Token: '▁'             | ID: 3
  Token: 'i'             | ID: 8
  Token: 's'             | ID: 5
  Token: '▁'             | ID: 3
  Token: 'a'             | ID: 6
  Token: '▁'             | ID: 3
  Token: 'b'             | ID: 33
  Token: 'r'             | ID: 7
  Token: 'a'             | ID: 6
  Token: 'n'             | ID: 10
  Token: 'c'           

### 5.3. Part-of-Speech (POS) Tagging on Sample Text
Using target text: *"The young student is reading an interesting book in the library."*
* **a. Using spaCy**
* **b. Using NLTK**

In [7]:
# 5.3.a POS Tagging using spaCy
import spacy

nlp = spacy.load("en_core_web_sm")
sentence = "The young student is reading an interesting book in the library."
doc = nlp(sentence)

print("--- 5.3.a POS Tagging using spaCy ---")
print(f"{'Token':<15} | {'POS Tag':<8} | {'Description'}")
print("-" * 60)
seen_tokens = set()
for token in doc:
    if token.text.lower() not in seen_tokens:
        seen_tokens.add(token.text.lower())
        print(f"{token.text:<15} | {token.pos_:<8} | {spacy.explain(token.pos_)}")

--- 5.3.a POS Tagging using spaCy ---
Token           | POS Tag  | Description
------------------------------------------------------------
The             | DET      | determiner
young           | ADJ      | adjective
student         | NOUN     | noun
is              | AUX      | auxiliary
reading         | VERB     | verb
an              | DET      | determiner
interesting     | ADJ      | adjective
book            | NOUN     | noun
in              | ADP      | adposition
library         | NOUN     | noun
.               | PUNCT    | punctuation


In [19]:
# 5.3.b POS Tagging using NLTK
import nltk
from nltk.tokenize import word_tokenize

# Ensure the required resources are downloaded
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

tokens_nltk = word_tokenize(sentence)
pos_tags_nltk = nltk.pos_tag(tokens_nltk)

# Direct mapping of Penn Treebank tags for reliable descriptions
penn_tags_map = {
    'CC': 'Coordinating conjunction',
    'CD': 'Cardinal number',
    'DT': 'Determiner',
    'EX': 'Existential there',
    'FW': 'Foreign word',
    'IN': 'Preposition or subordinating conjunction',
    'JJ': 'Adjective',
    'JJR': 'Adjective, comparative',
    'JJS': 'Adjective, superlative',
    'LS': 'List item marker',
    'MD': 'Modal',
    'NN': 'Noun, singular or mass',
    'NNS': 'Noun, plural',
    'NNP': 'Proper noun, singular',
    'NNPS': 'Proper noun, plural',
    'PDT': 'Predeterminer',
    'POS': 'Possessive ending',
    'PRP': 'Personal pronoun',
    'PRP$': 'Possessive pronoun',
    'RB': 'Adverb',
    'RBR': 'Adverb, comparative',
    'RBS': 'Adverb, superlative',
    'RP': 'Particle',
    'SYM': 'Symbol',
    'TO': 'to',
    'UH': 'Interjection',
    'VB': 'Verb, base form',
    'VBD': 'Verb, past tense',
    'VBG': 'Verb, gerund or present participle',
    'VBN': 'Verb, past participle',
    'VBP': 'Verb, non-3rd person singular present',
    'VBZ': 'Verb, 3rd person singular present',
    'WDT': 'Wh-determiner',
    'WP': 'Wh-pronoun',
    'WP$': 'Possessive wh-pronoun',
    'WRB': 'Wh-adverb',
    '.': 'Punctuation mark',
    ',': 'Punctuation mark'
}

print("--- 5.3.b POS Tagging using NLTK ---")
print(f"{'Token':<15} | {'POS Tag':<8} | {'Description'}")
print("-" * 60)
seen_tokens_nltk = set()
for token, tag in pos_tags_nltk:
    if token.lower() not in seen_tokens_nltk:
        seen_tokens_nltk.add(token.lower())
        desc = penn_tags_map.get(tag, 'N/A')
        print(f"{token:<15} | {tag:<8} | {desc}")

--- 5.3.b POS Tagging using NLTK ---
Token           | POS Tag  | Description
------------------------------------------------------------
The             | DT       | Determiner
young           | JJ       | Adjective
student         | NN       | Noun, singular or mass
is              | VBZ      | Verb, 3rd person singular present
reading         | VBG      | Verb, gerund or present participle
an              | DT       | Determiner
interesting     | JJ       | Adjective
book            | NN       | Noun, singular or mass
in              | IN       | Preposition or subordinating conjunction
library         | NN       | Noun, singular or mass
.               | .        | Punctuation mark


### 5.4. POS Tagging and Frequency on Corpus using spaCy
We will perform POS tagging on the dataset `/content/input_sub_word_data.txt` and display unique tokens, tags, descriptions, and token frequencies.

In [9]:
from collections import Counter

doc_corpus = nlp(input_text)

# Count frequency of each token text
token_freqs = Counter([token.text.lower() for token in doc_corpus if not token.is_space])

seen_corpus_tokens = set()

print("--- 5.4. POS Tagging with Frequency (spaCy) ---")
print(f"{'Token':<15} | {'POS Tag':<8} | {'Frequency':<10} | {'Description'}")
print("-" * 70)
for token in doc_corpus:
    if token.is_space:
        continue
    token_lower = token.text.lower()
    if token_lower not in seen_corpus_tokens:
        seen_corpus_tokens.add(token_lower)
        freq = token_freqs[token_lower]
        print(f"{token.text:<15} | {token.pos_:<8} | {freq:<10} | {spacy.explain(token.pos_)}")

--- 5.4. POS Tagging with Frequency (spaCy) ---
Token           | POS Tag  | Frequency  | Description
----------------------------------------------------------------------
Natural         | ADJ      | 6          | adjective
language        | NOUN     | 12         | noun
processing      | NOUN     | 7          | noun
is              | AUX      | 14         | auxiliary
a               | DET      | 28         | determiner
branch          | NOUN     | 1          | noun
of              | ADP      | 22         | adposition
artificial      | ADJ      | 1          | adjective
intelligence    | NOUN     | 1          | noun
that            | PRON     | 4          | pronoun
enables         | VERB     | 1          | verb
computers       | NOUN     | 1          | noun
to              | PART     | 15         | particle
understand      | VERB     | 4          | verb
,               | PUNCT    | 76         | punctuation
interpret       | VERB     | 1          | verb
and             | CCONJ    | 27   

### 5.5, 5.6 & 5.7 Word2Vec Models (CBOW, Skip-Gram, & Cosine Similarity)
First, we'll write the input corpus to `corpus.txt` and train both CBOW and Skip-Gram models using `gensim`.

In [10]:
corpus_content = """Natural language processing is interesting
Natural language processing is useful
I love machine learning
Machine learning is useful
Deep learning is a part of machine learning
Artificial intelligence is changing the world
Natural language processing uses machine learning
Deep learning and artificial intelligence are related
I study natural language processing
Machine learning helps artificial intelligence"""

with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write(corpus_content)

# Preprocess corpus for gensim
corpus_sentences = [line.lower().split() for line in corpus_content.strip().split('\n')]
print("Prepared Sentences:", len(corpus_sentences))

Prepared Sentences: 10


In [11]:
# 5.5. CBOW Model Training
from gensim.models import Word2Vec

# sg=0 specifies CBOW
model_cbow = Word2Vec(sentences=corpus_sentences, vector_size=50, window=3, min_count=1, sg=0, epochs=150)

selected_word = "learning"
vector_cbow = model_cbow.wv[selected_word]
similar_cbow = model_cbow.wv.most_similar(selected_word, topn=3)

print("--- 5.5. CBOW Word2Vec Model ---")
print(f"Vector representation of '{selected_word}':\n{vector_cbow[:10]}... (truncated)")
print(f"Three most similar words to '{selected_word}':")
for w, sim in similar_cbow:
    print(f"  {w:<15} | Similarity: {sim:.4f}")

--- 5.5. CBOW Word2Vec Model ---
Vector representation of 'learning':
[-0.00292222  0.00130013  0.00950361  0.02088834 -0.02022711 -0.01920285
  0.01486764  0.02278602 -0.01447753 -0.01168053]... (truncated)
Three most similar words to 'learning':
  are             | Similarity: 0.3682
  the             | Similarity: 0.3678
  is              | Similarity: 0.3205


In [12]:
# 5.6. Skip-Gram Model Training
# sg=1 specifies Skip-Gram
model_sg = Word2Vec(sentences=corpus_sentences, vector_size=50, window=3, min_count=1, sg=1, epochs=150)

target_word = "language"
vector_sg = model_sg.wv[target_word]
similar_sg = model_sg.wv.most_similar(target_word, topn=5)

print("--- 5.6. Skip-Gram Word2Vec Model ---")
print(f"Vector representation of '{target_word}':\n{vector_sg[:10]}... (truncated)")
print(f"Five most similar words to '{target_word}':")
for w, sim in similar_sg:
    print(f"  {w:<15} | Similarity: {sim:.4f}")

--- 5.6. Skip-Gram Word2Vec Model ---
Vector representation of 'language':
[-0.00187105  0.00624846 -0.01452964 -0.00074477  0.0138446   0.0112696
 -0.00518073  0.00953798 -0.02003964  0.00926343]... (truncated)
Five most similar words to 'language':
  is              | Similarity: 0.3110
  related         | Similarity: 0.2896
  artificial      | Similarity: 0.2733
  machine         | Similarity: 0.2386
  a               | Similarity: 0.2312


In [13]:
# 5.7. Cosine Similarity on Select Word Pairs (using the CBOW model as base)
pairs = [
    ("machine", "learning"),
    ("natural", "language"),
    ("deep", "learning")
]

print("--- 5.7. Cosine Similarity Results ---")
for w1, w2 in pairs:
    similarity = model_cbow.wv.similarity(w1, w2)
    print(f"Similarity between '{w1}' and '{w2}': {similarity:.4f}")

--- 5.7. Cosine Similarity Results ---
Similarity between 'machine' and 'learning': 0.2150
Similarity between 'natural' and 'language': 0.1930
Similarity between 'deep' and 'learning': 0.1147
